# 循環神經網路

## 沒有記憶功能的非循環神經網路

函式 f(x) 每次被呼叫時，第一行永遠是 y = 0

這意味著：無論你上一秒輸入的是 $2$ 還是 $100$，這一次的運算都從零開始。

它不關心過去（History-independent），這就是非循環神經網路（Non-recurrent Neural Network） 的核心特性。

In [47]:
def f(x):
    y = 0
    y += x ** 2
    return y

print(f(2), '\t', f(3))
print(f(3), '\t', f(4))

4 	 9
9 	 16


## 沒有記憶功能的非循環神經網路

這段程式碼模擬了 RNN 的簡化數學模型： $y_{t}, h_{t} = f(x_{t}, h_{t-1})$

隱藏層公式： $h_t = h_{t-1} + w_x \cdot x_t$，權重設定為 $w_x = 2$。

輸出層公式： $y_t = h_t + f(x_t)$，$f(x_t) = x^2$。

這種結構讓網路能夠處理序列型數據（如文字、股價、語音）。例如在翻譯時，網路必須記得前面的單字是「我」，後面的動詞才會選擇對應的變化。

In [48]:
class rf():
    def __init__(self):
        self.h = 0

    def forward(self, x):
        self.h += 2 * x
        return self.h + x ** 2
    
    def __call__(self, x):
        return self.forward(x)
    
f = rf()
print(f(2), '\t', f(3))
print(f(3), '\t', f(2))

8 	 19
25 	 24


## 初始化模型參數

In [49]:
import numpy as np

np.random.seed(1)

def rnn_params_init(input_dim, hidden_dim, output_dim, scale = 0.01):        
    # 初始化 Input 到 Hidden 層的權重 (Wx)
    Wx = np.random.randn(input_dim, hidden_dim) * scale 
    
    # 初始化 Hidden 到 Hidden 層的權重 (Wh)，也就是上一秒跟這一秒的連結
    Wh = np.random.randn(hidden_dim, hidden_dim) * scale 
    
    # 初始化 Hidden 層的偏置值 (Bias)，先整把塞 0 給它
    bh = np.zeros((1, hidden_dim)) 

    # 初始化 Hidden 到 Output 層的權重 (Wf)
    Wf = np.random.randn(hidden_dim, output_dim) * scale 
    
    # 初始化 Output 層的偏置值 (Bias)
    bf = np.zeros((1, output_dim)) 

    # 把這包參數封裝好丟回去
    return [Wx, Wh, bh, Wf, bf]

def rnn_hidden_state_init(batch_dim, hidden_dim):
    # 幫 Hidden state 挖一個空位，初始值一樣先全給 0
    return np.zeros((batch_dim, hidden_dim))

In [64]:
import numpy as np

np.random.seed(1)

# 生成 5 個時刻（Time Steps），每批只有 1 個樣本（Batch Size）的一組測試資料
# 我們定義一個 RNN 模型，規格如下：
# Input Dimension: 4 (輸入維度)
# Hidden Dimension: 10 (隱藏層神經元數量)
# Output Dimension: 4 (輸出維度)
if True:
    T = 5
    input_dim, hidden_dim, output_dim = 4, 10, 4
    batch_size = 1
    seq_len = 5
    
    # 隨機生一組輸入 Xs，形狀是 (序列長度, 批次大小, 特徵維度)
    # 這就是我們要餵給模型吃的「生資料」
    Xs = np.random.rand(seq_len, batch_size, input_dim)
    
    # 隨機生一組正確答案 Ys，通常是用來算 Loss 的「標籤 (Label)」
    # 這裡是用整數，看起來是準備要做分類任務 (Classification)
    Ys = np.random.randint(input_dim, size = (seq_len, batch_size))
    
# 把這兩組資料印出來瞧瞧，看看形狀對不對
print("這是輸入資料 Xs：\n", Xs)
print("這是對應的標籤 Ys：\n", Ys)

這是輸入資料 Xs：
 [[[4.17022005e-01 7.20324493e-01 1.14374817e-04 3.02332573e-01]]

 [[1.46755891e-01 9.23385948e-02 1.86260211e-01 3.45560727e-01]]

 [[3.96767474e-01 5.38816734e-01 4.19194514e-01 6.85219500e-01]]

 [[2.04452250e-01 8.78117436e-01 2.73875932e-02 6.70467510e-01]]

 [[4.17304802e-01 5.58689828e-01 1.40386939e-01 1.98101489e-01]]]
這是對應的標籤 Ys：
 [[1]
 [1]
 [1]
 [3]
 [3]]


In [ ]:
# 1. 先生出模型參數，這就像是把模型的「骨架」架起來
params = rnn_params_init(input_dim, hidden_dim, output_dim)

print("參數的形狀：", [p.shape for p in params])  # 把這包參數印出來看看，確保它們的形狀跟我們預期的一樣

# 2. 挖一個 Hidden state 的初始空位，這格通常全填 0
H_0 = rnn_hidden_state_init(batch_size, hidden_dim)

## 向前傳播

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

$$y_t = W_y h_t$$

In [50]:
def rnn_forward(params, Xs, H_):
    # 先把那包 params 解壓縮出來，不然下面寫起來很痛苦
    Wx, Wh, bh, Wf, bf = params
    
    # 把初始的隱藏狀態 (Hidden State) 拿過來接
    H = H_ #np.copy(H_)   
   
    Fs = []        # 用來存每一格產出的 Output
    Hs = {}        # 用來存每一格產出的 Hidden State，方便之後做 Backprop
    
    # 稍微留一下底，把「前一時刻」的狀態（也就是 -1）先存進去
    Hs[-1] = np.copy(H)    
 
    # 開始跑 Time Steps 的迴圈，Xs 有多長就跑幾次
    for t in range(len(Xs)):
        X = Xs[t]       # 抓出當前時間點的輸入資料
        
        # 這是 RNN 的心臟：把目前的輸入、前一格的狀態加權之後，丟進 tanh 擠壓一下
        H = np.tanh(np.dot(X, Wx) + np.dot(H, Wh) + bh)
        
        # 根據算好的 H，再過一層線性轉換算出最後的輸出結果 F
        F = np.dot(H, Wf) + bf       

        # 把這一步算出來的結果通通塞進 list 跟 dict 裡面
        Fs.append(F)
        Hs[t] = H
        
    # 最後把這一整串算出來的 Fs 跟 Hs 吐回去
    return Fs, Hs

In [51]:
def rnn_forward_step(params, X, preH):
    # 一樣先把那包參數拆開來用，省得後面寫得落落長
    Wx, Wh, bh, Wf, bf = params     
    
    # 這是 RNN 的靈魂公式：把「目前的 Input」跟「上一動的 Hidden State」加權揉在一起
    # 再過一個 tanh 激活函數，算出這一動的 H
    H = np.tanh(np.dot(X, Wx) + np.dot(preH, Wh) + bh)
    
    # 用剛剛算出來的新 H，直接推一把算出這一格的預測輸出 F
    F = np.dot(H, Wf) + bf 
    
    # 把這一步的結果跟狀態一起吐回去，下一格還要接著用
    return F, H

In [52]:
def rnn_forward_(params, Xs, H_):
    # 一樣先把那包參數拆開，省得後面寫得落落長
    Wx, Wh, bh, Wf, bf = params
    
    # 這裡的 H 是初始狀態，先接過來
    H = H_  
   
    Fs = []        # 準備拿來裝每一動的輸出 (Output)
    Hs = {}        # 準備拿來裝每一動的隱藏狀態 (Hidden State)
    
    # 稍微留個底，把「前一時刻」的狀態（也就是 -1）先 copy 一份存起來
    Hs[-1] = np.copy(H)    
 
    # 這裡開始跑 Time Steps 的迴圈
    for t in range(len(Xs)):
        X = Xs[t]       
        
        # 直接呼叫剛才寫好的 step function，不用在這邊重複造輪子
        # 算完之後，把新的 H 覆蓋掉舊的，餵給下一格
        F, H = rnn_forward_step(params, X, H)       
        
        # 把算好的結果通通塞進 list 跟 dict 裡面
        Fs.append(F)
        Hs[t] = H
        
    # 最後把這一整串算好的 Fs 跟 Hs 吐回去就收工了
    return Fs, Hs

In [71]:
# 3. 跑一次 Forward Pass，看看資料餵進去後，能不能順利吐出預測值 Fs 跟隱藏狀態 Hs
Fs, Hs = rnn_forward(params, Xs, H_0) 

print("預測值:", Fs)
print("正確答案:", Ys)

# 5. 先印一下 shape（形狀），確保預測值跟正確答案的維度是對的
print("預測值的 Shape:", Fs[0].shape, "正確答案的 Shape:", Ys[0].shape)

預測值: [array([[-0.00020459, -0.00038285,  0.00032914, -0.00045276]]), array([[-7.65979897e-05, -2.29546905e-04,  1.28798424e-04,
        -4.59341215e-05]]), array([[-0.00021536, -0.00059076,  0.00038169, -0.00029498]]), array([[-0.00034543, -0.00058648,  0.00046515, -0.00071175]]), array([[-0.00011952, -0.00030863,  0.00026055, -0.00025836]])]
正確答案: [[1]
 [1]
 [1]
 [3]
 [3]]
預測值的 Shape: (1, 4) 正確答案的 Shape: (1,)


## 損失函數

In [53]:
def softmax(Z):
    # 對輸入的 Z 做 softmax，先減掉最大值避免數值爆掉
    A = np.exp(Z - np.max(Z, axis=-1, keepdims=True))
    # 將指數結果做正規化，讓每一列加起來等於 1
    return A / np.sum(A, axis=-1, keepdims=True)

def softmax_cross_entropy(Z, y, onehot=False):
    # 樣本的總數（有幾筆資料）
    m = len(Z)

    # 先把模型輸出 Z 丟進 softmax，算出每一類的機率
    F = softmax(Z)

    if onehot:
        # 如果 y 是 one-hot 編碼
        # 直接用 cross entropy 的公式算 loss
        loss = -np.sum(y * np.log(F)) / m
    else:
        # 如果 y 是類別的 index（例如 0、1、2）
        # 把 y 攤平成一維（但這行實際上不會影響後面）
        y.flatten()

        # 取出每筆資料「正確類別」對應的機率，再取 log
        log_Fy = -np.log(F[range(m), y])

        # 把所有樣本的 loss 加起來，再取平均
        loss = np.sum(log_Fy) / m

    return loss

def cross_entropy_grad(Z, Y, onehot=False, softmax_out=False):
    # 判斷輸入的 Z 是不是已經是 softmax 的輸出
    if softmax_out:
        F = Z
    else:
        # 如果不是，就先做一次 softmax
        F = softmax(Z)

    if onehot:
        # 如果 Y 是 one-hot 編碼，直接用 F - Y
        grad = (F - Y) / len(Z)
    else:
        # 如果 Y 是類別索引（例如 [0, 2, 1]）
        m = len(Y)          # 樣本數
        grad = F.copy()       # 複製一份避免動到原本的 F
        # 對正確類別的位置減 1
        grad[np.arange(m), Y] -= 1
        # 對 batch size 做平均
        grad /= m

        # 以下是另一種寫法，這裡先註解起來
        # I_i = np.zeros_like(Z)
        # I_i[np.arange(len(Z)),Y] = 1
        # return (F - I_i) /len(Z)  #Z.shape[0]

    # 回傳對 Z 的梯度
    return grad

# def cross_entropy_grad_loss(F, y, softmax_out=False, onehot=False):
#     # 判斷輸入的 F 是否已經經過 Softmax 處理
#     if softmax_out:
#         # 如果已經是機率分布了，直接丟進去算 Loss
#         loss = softmax_cross_entropy(F, y, onehot)
#         # 註解掉的這行可能是想區分一般的 Cross Entropy 與 Softmax 版
#         # loss = cross_entropy_loss(F, y, onehot)
#     else:
#         # 如果 F 還是 Logits（原始輸出），則執行含 Softmax 的 Cross Entropy 計算
#         loss = softmax_cross_entropy(F, y, onehot)

#     # 計算梯度（Grad），同樣會根據 softmax_out 決定內部是否要補做 Softmax
#     grad = cross_entropy_grad(F, y, onehot, softmax_out)

#     # 同時回傳損失值與梯度，方便後向傳播使用
#     return loss, grad

def cross_entropy_grad_loss(Z, y, softmax_out=False, onehot=False):
    """
    優化後的函式：共用 Softmax 運算結果，同時回傳 Loss 與 Gradient。
    """
    # 1. 決定機率分布 F (Softmax 的結果)
    if softmax_out:
        # 如果已經是 Softmax 輸出，直接共用
        F = Z
    else:
        # 如果是 Logits，算一次 Softmax 就好，後面大家都用這份 F
        F = softmax(Z)

    # 2. 計算 Loss
    # 注意：這裡我們傳入已經算好的 F，並告訴後面的函式 softmax_out=True
    # (假設你原有的 softmax_cross_entropy 有支援這個判斷，或是我們直接在這裡算)
    m = len(y)
    if onehot:
        loss = -np.sum(y * np.log(F + 1e-12)) / m
    else:
        # 這裡示範直接計算，減少函式呼叫的開銷
        loss = -np.sum(np.log(F[np.arange(m), y] + 1e-12)) / m

    # 3. 計算梯度 (Gradient)
    # 直接利用剛剛算好的 F 進行計算，效能最優
    grad = F.copy()
    if onehot:
        grad = (grad - y) / m
    else:
        grad[np.arange(m), y] -= 1
        grad /= m

    return loss, grad

In [54]:
def numerical_gradient_from_df(f, p, df, h=1e-5):
  # 建立一個與 p 形狀相同、內容全為 0 的陣列，用來存每個參數的梯度
  grad = np.zeros_like(p)

  # 使用 nditer 逐一走訪 p 中的每一個元素（支援多維陣列）
  it = np.nditer(p, flags=['multi_index'], op_flags=['readwrite'])

  # 只要 iterator 還沒跑完就持續計算
  while not it.finished:
    # 取得目前走訪到的索引位置
    idx = it.multi_index

    # 先把原本的參數值存起來
    oldval = p[idx]

    # 將該參數往正方向微調一點
    p[idx] = oldval + h
    pos = f()       # 在參數被改動後重新呼叫 f()，取得正方向的輸出結果

    # 將該參數往負方向微調一點
    p[idx] = oldval - h
    neg = f()       # 在參數被改動後重新呼叫 f()，取得負方向的輸出結果

    # 將參數值還原成原本的狀態，避免影響下一次計算
    p[idx] = oldval

    # 使用中央差分法計算梯度，並與上游傳下來的 df 做加權
    grad[idx] = np.sum((pos - neg) * df) / (2 * h)

    # 另一種寫法（使用內積），目前被註解掉
    # grad[idx] = np.dot((pos - neg), df) / (2 * h)

    # 移動到下一個參數位置
    it.iternext()

  # 回傳整個參數 p 的數值梯度
  return grad

# 通用數值梯度函式
def numerical_gradient(f, params, eps=1e-6):
    numerical_grads = []  # 用來存每一個參數對應的梯度

    for x in params:
        # x 可能是多維陣列，這裡會針對 x 裡面的每一個元素計算數值梯度
        grad = np.zeros(x.shape)  # 建立一個跟 x 形狀一樣的陣列來存梯度

        # 使用 nditer 逐一走訪 x 裡面的每個元素
        it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])

        while not it.finished:
            idx = it.multi_index   # 目前元素的索引位置

            old_value = x[idx]     # 先把原本的值存起來

            x[idx] = old_value + eps  # 對目前的值加上一個很小的 eps
            fx = f()                  # 計算 f(x + eps) 的結果

            x[idx] = old_value - eps  # 對目前的值減去一個很小的 eps
            fx_ = f()                 # 計算 f(x - eps) 的結果

            # 使用中央差分法計算數值梯度
            grad[idx] = (fx - fx_) / (2 * eps)

            x[idx] = old_value        # 記得把參數值還原，避免影響後續計算
            it.iternext()             # 移動到下一個元素

        numerical_grads.append(grad)  # 將目前參數的梯度存起來

    return numerical_grads            # 回傳所有參數的梯度

# def f():
#     return compute_loss_reg(forward_propagation, softmax_cross_entropy_reg, X, y, parameters)

In [55]:
def rnn_loss_grad(Fs, Ys, loss_fn = cross_entropy_grad_loss, flatten = True):   
    # 先把總 Loss 歸零，等等一格一格加回來
    loss = 0
    # 用來存每一格對應的 Gradient (dF)，準備拿來做 Backprop
    dFs = {}
   
    # 開始跑迴圈，把預測值 Fs 跟答案 Ys 對齊來算帳
    for t in range(len(Fs)):
        F = Fs[t]
        Y = Ys[t]   
        
        # 這裡做個簡單的防呆：如果需要把答案拉平（Flatten），而且維度太高就處理一下
        # 避免後面的 Loss function 噴錯 (Error)
        if flatten and Y.ndim >= 2:          
            Y = Y.flatten()
            
        # 丟進 Loss function 算一下這一格噴了多少 Loss，順便把 dF 吐出來
        loss_t, dF_t = loss_fn(F, Y)
        
        # 把這一格的 Loss 累加進去總分裡
        loss += loss_t        
        
        # 把這格的 Gradient 存起來，晚點反向傳播（BPTT）會用到
        dFs[t] = dF_t
       
    # 最後把總 Loss 跟那一包 dFs 丟回去
    return loss, dFs

## 向後傳播

In [56]:
import math

# 避免梯度爆炸的防護罩：把過大的梯度縮小回合理的範圍
def grad_clipping(grads, alpha):
    # 先算一下這一整包梯度的 L2 Norm (長度)
    norm = math.sqrt(sum((grad ** 2).sum() for grad in grads))
    # 如果長度超過我們設定的門檻 (alpha)，就按比例縮小
    if norm > alpha:
        ratio = alpha / norm
        for i in range(len(grads)):
            # 把它「壓回去」，不要讓更新步長大到失控
            grads[i] *= ratio 
            
def rnn_backward(params, Xs, Hs, dZs, clip_value = 5.):    
    # 一樣先把參數解開來，等等算微分比較好對名字
    Wx, Wh, bh, Wf, bf = params
    
    # 準備拿來存各個參數的梯度，先用 zeros_like 挖好同樣形狀的坑
    dWx, dWh, dWf = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(Wf)
    dbh, dbf = np.zeros_like(bh), np.zeros_like(bf)       

    # 這一格的梯度會傳給「上一動」，所以要有一個 dh_next 接著
    dh_next = np.zeros_like(Hs[0])
    h = Hs
    x = Xs
    
    T = len(Xs)  # 序列長度，也就是總共跑了幾個 Time Step
    
    # RNN 的精華：從最後一個時刻往回推 (Backprop Through Time)
    for t in reversed(range(T)): 
        dZ = dZs[t]        # 抓出這一格的輸出誤差
        
        # 算一下對 Output 層權重 (Wf) 的梯度
        dWf += np.dot(h[t].T, dZ)
        # 算一下對 Output 層 Bias (bf) 的梯度
        dbf += np.sum(dZ, axis=0, keepdims=True)         
        
        # 梯度往回傳：當下的誤差 + 從「未來」傳回來的梯度
        dh = np.dot(dZ, Wf.T) + dh_next 
        
        # 通過 tanh 的微分 (1 - h^2)，把梯度傳過啟動函數
        dZh = (1 - h[t] * h[t]) * dh 
        
        # 算一下這一格對隱藏層參數 (bh, Wx, Wh) 的貢獻
        dbh += np.sum(dZh, axis=0, keepdims=True) 
        dWx += np.dot(x[t].T, dZh)
        # 這裡會用到 t-1 的隱藏狀態，這就是為什麼 Hs 要存一整包的原因
        dWh += np.dot(h[t-1].T, dZh)
        
        # 把梯度傳給下一個（其實是前一個時刻）的 dh_next
        dh_next = np.dot(dZh, Wh.T)
   
    # 把這堆算好的梯度打包
    grads = [dWx, dWh, dbh, dWf, dbf]
    
    # 做個檢查，如果有設門檻就做 Clipping，避免梯度炸掉
    if clip_value is not None:
        grad_clipping(grads, clip_value)
        
    # 大功告成，把這包梯度吐回去準備更新權重
    return grads

In [57]:
def rnn_backward_step(params, dZ, X, H, H_, dh_next): 
    # 把那幾組權重參數先拆出來用
    Wx, Wh, bh, Wf, bf = params
    
    # 算一下這一動的輸出權重梯度 (dWf)
    dWf = np.dot(H.T, dZ)

    # 算一下 Output 層的 Bias 梯度 (dbf)
    dbf = np.sum(dZ, axis=0, keepdims=True)         
    
    # 關鍵：這一層的梯度 (dh) 是「目前的輸出誤差」加上「從未來傳回來的梯度」
    dh = np.dot(dZ, Wf.T) + dh_next 
    
    # 過一下 tanh 的微分，把梯度傳進去隱藏層核心 (dZh)
    dZh = (1 - H * H) * dh 

    # 算一下隱藏層各個參數的梯度 (Bias, Input Weight, Hidden Weight)
    dbh = np.sum(dZh, axis=0, keepdims=True) 
    dWx = np.dot(X.T, dZh)
    # 這裡的 H_ 代表上一動的 Hidden State，用來算 Wh 的梯度
    dWh = np.dot(H_.T, dZh)
    
    # 算出要再往「過去」傳的梯度 (dh_next)，準備交棒給上一個時刻
    dh_next = np.dot(dZh, Wh.T)
    
    # 把這堆算好的梯度通通回傳，收工！
    return dWx, dWh, dbh, dWf, dbf, dh_next

In [58]:
def rnn_backward_(params, Xs, Hs, dZs, clip_value = 5.): 
    # 一樣先把參數拆一拆，順便把累加梯度的「坑」先挖好
    Wx, Wh, bh, Wf, bf = params
    dWx, dWh, dWf = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(Wf)
    dbh, dbf = np.zeros_like(bh), np.zeros_like(bf)
    
    # 準備一個 dh_next 來接住從「未來時刻」傳回來的梯度
    dh_next = np.zeros_like(Hs[0])
    
    # 算一下總共跑了幾格時間序列
    T = len(Xs)  
    
    # 這裡開始「倒帶」跑迴圈，從最後一秒往第一秒推回去
    for t in reversed(range(T)):  
        dZ = dZs[t]     # 這格的輸出誤差
        H = Hs[t]       # 這格的隱藏狀態
        H_ = Hs[t-1]    # 前一格的隱藏狀態（這就是 RNN 記住過去的關鍵）
        X = Xs[t]       # 這格的輸入資料
        
        # 直接呼叫剛剛寫好的單步 backward function，把這格產生的梯度算出來
        # 這裡會把 dh_next 一直傳下去，像接力賽一樣
        dWx_, dWh_, dbh_, dWf_, dbf_, dh_next = rnn_backward_step(params, dZ, X, H, H_, dh_next)
        
        # 把算出來的梯度「疊加」上去，因為同一個參數在每個 Time Step 都會貢獻一點梯度
        for grad, grad_t in zip([dWx, dWh, dbh, dWf, dbf], [dWx_, dWh_, dbh_, dWf_, dbf_]):
            grad += grad_t      

    # 把疊加完的梯度整包封裝起來
    grads = [dWx, dWh, dbh, dWf, dbf]
    
    # 如果梯度太狂、暴衝了，就用 Clipping 把它壓回正常值
    if clip_value is not None:
        grad_clipping(grads, clip_value)
    
    # 把這包心血結晶吐回去，準備拿去更新權重
    return grads

## 梯度驗證

In [63]:
# -------- 檢查梯度 (Gradient Check) 的前置作業 -------------   

# 1. 先生出模型參數，這就像是把模型的「骨架」架起來
params = rnn_params_init(input_dim, hidden_dim, output_dim)

print("參數的形狀：", [p.shape for p in params])  # 把這包參數印出來看看，確保它們的形狀跟我們預期的一樣

# 2. 挖一個 Hidden state 的初始空位，這格通常全填 0
H_0 = rnn_hidden_state_init(batch_size, hidden_dim)

# 3. 跑一次 Forward Pass，看看資料餵進去後，能不能順利吐出預測值 Fs 跟隱藏狀態 Hs
Fs, Hs = rnn_forward(params, Xs, H_0) 

# 4. 指定我們要用的 Loss function
loss_function = rnn_loss_grad

# 5. 先印一下 shape（形狀），確保預測值跟正確答案的維度是對的
print("預測值的 Shape:", Fs[0].shape, "正確答案的 Shape:", Ys[0].shape)

# 6. 算出這一次 Forward 有多少 Loss，順便拿到對輸出的梯度 dFs
loss, dFs = loss_function(Fs, Ys)  

# 7. 最後把這包 dFs 往回丟，算出一整組參數的梯度 grads，準備拿去跟數值梯度比對
grads = rnn_backward(params, Xs, Hs, dFs)

這是輸入資料 Xs：
 [[[0.5881308  0.89771373 0.89153073 0.81583748]]

 [[0.03588959 0.69175758 0.37868094 0.51851095]]

 [[0.65795147 0.19385022 0.2723164  0.71860593]]

 [[0.78300361 0.85032764 0.77524489 0.03666431]]

 [[0.11669374 0.7512807  0.23921822 0.25480601]]]
這是對應的標籤 Ys：
 [[3]
 [1]
 [1]
 [3]
 [1]]
參數的形狀： [(4, 10), (10, 10), (1, 10), (10, 4), (1, 4)]
預測值的 Shape: (1, 4) 正確答案的 Shape: (1,)


In [60]:
def rnn_loss():
    # 初始隱藏狀態先給它一排 0
    H_0 = np.zeros((1, hidden_dim))
    H = np.copy(H_0)
    
    # 跑一次前向傳播 (Forward Pass)
    Fs, Hs = rnn_forward(params, Xs, H) 
    
    # 指定計算 Loss 的 function
    loss_function = rnn_loss_grad
    
    # 把 Loss 算出來，dFs 在這裡先放生它，我們只需要 loss 值
    loss, dFs = loss_function(Fs, Ys)     
    return loss

# 呼叫工具來跑「數值梯度」，原理是把參數微調一點點 (1e-6) 來暴力計算斜率
# 這是用來檢驗我們手寫的 rnn_backward 對不對的「標準答案」
numerical_grads = numerical_gradient(rnn_loss, params, 1e-6)

# 定義誤差計算方式（相對誤差），避免因為數值太小導致誤判
# 台灣工程師常用 lambda 寫這種一行的工具 function，乾淨俐落
diff_error = lambda x, y: np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

print("目前的 Loss 值:", loss)
print("各個參數 [dWx, dWh, dbh, dWf, dbf] 的梯度誤差：")

# 把手寫梯度 (grads) 跟數值梯度 (numerical_grads) 拿出來一對一「釘孤支」
for i in range(len(grads)):
    # 這裡算出來的誤差如果大於 1e-5，通常代表你的 Backward 邏輯「噴了」
    print(f"參數 {i} 的誤差: {diff_error(grads[i], numerical_grads[i])}")

# 隨便挑幾個數值印出來肉眼檢查一下，看看正負號跟位數有沒有對齊
print("手寫梯度前兩項:", grads[1][:2])
print("數值梯度前兩項:", numerical_grads[1][:2])

目前的 Loss 值: 6.931604253096048
各個參數 [dWx, dWh, dbh, dWf, dbf] 的梯度誤差：
參數 0 的誤差: 3.102288266166044e-06
參數 1 的誤差: 6.606174384443744e-05
參數 2 的誤差: 8.42115646591704e-07
參數 3 的誤差: 2.8703402385675273e-07
參數 4 的誤差: 3.19253887743486e-10
手寫梯度前兩項: [[-2.39049602e-04  8.14220495e-05  1.57776751e-04  5.67414815e-05
  -2.52527076e-04  7.67751376e-05  8.81253550e-05  2.07270381e-04
  -6.92579913e-05  5.33532921e-05]
 [-1.59775181e-04  8.33693576e-05  7.68434971e-05  4.16925859e-05
  -1.31768112e-04  1.87065893e-05  3.02967764e-05  1.17071893e-04
  -3.32692578e-05  2.22690120e-05]]
數值梯度前兩項: [[-2.39049225e-04  8.14224244e-05  1.57776903e-04  5.67417224e-05
  -2.52526888e-04  7.67750308e-05  8.81250628e-05  2.07270645e-04
  -6.92579327e-05  5.33528777e-05]
 [-1.59774860e-04  8.33697555e-05  7.68438646e-05  4.16924273e-05
  -1.31768374e-04  1.87072580e-05  3.02975423e-05  1.17071686e-04
  -3.32689432e-05  2.22688534e-05]]
